[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ersilia-os/ub-cedd-projects-workshop/blob/main/projects/yellow/notebooks/yellow_baseline_models.ipynb)

# Training baseline models for ACE1 inhibition

**Yellow group · Hypertension**

This notebook trains models that say whether a molecule inhibits ACE1. It builds a random
forest on fingerprints by hand, then lets **LazyQSAR** try several molecular representations
and models automatically, and compares the two on the same hard split. The better of the two
is trained on all the data and saved, ready to point at indoles and xanthones.

## What you will do

- Look at the data and clean it: drop the few molecules too large to model sensibly.
- Turn every molecule into a fingerprint and train a random forest to predict active or
  inactive.
- Score it on a random split and on a harder scaffold split, with 5-fold cross-validation.
- Run LazyQSAR, which tries five molecular representations for you, and compare it with the
  random forest on the same split.
- Train the better model on all the data and download it.

## Setup

Run the cell below first. In Colab it downloads the workshop repository (including the data) and installs the packages this project needs. It takes about a minute. **Don't change it.**

In [ ]:
PROJECT = "yellow"
NEEDS_GPU = False
import os, sys, shutil, subprocess
if "google.colab" in sys.modules:
    repo_dir = "/content/ub-cedd-projects-workshop"
    if not os.path.exists(repo_dir):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ersilia-os/ub-cedd-projects-workshop.git", repo_dir], check=True)
    else:
        subprocess.run(["git", "-C", repo_dir, "pull", "--ff-only"], check=True)
    os.chdir(f"{repo_dir}/projects/{PROJECT}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
for _cached in [m for m in sys.modules if m == "scripts" or m.startswith("scripts.")]:
    del sys.modules[_cached]  # forget helper modules imported before the pull above
has_gpu = shutil.which("nvidia-smi") is not None and subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
print(f"Python {sys.version.split()[0]} | GPU: {'yes' if has_gpu else 'no'} | Folder: {os.getcwd()}")
if NEEDS_GPU and not has_gpu:
    print("WARNING: this notebook needs a GPU. Go to Runtime > Change runtime type, choose CPU, and run this cell again.")

## 1. Load the curated data

The first notebook turned four sources of ACE1 data into a single table, and the group
uploaded it to its Drive folder. It is copied into `data/` in this repository, so it is
already here.

Each row is one molecule. The columns we need are:

- `smiles`, the molecule itself.
- `activity`, the label: 1 for an inhibitor, 0 for not.
- `pactivity`, how strongly it inhibits, where higher means more potent. This notebook is
  about classification, so we only use it to look at the data.
- `evidence`, where the label came from: a real measurement, or only a limit such as "no
  inhibition at 100 uM".

In [ ]:
import numpy as np
import pandas as pd
import stylia
from scripts import modelling

RANDOM_SEED = 42
THRESHOLD = 6.0  # pActivity 6 is the 1 uM cutoff used to label the molecules

curated = pd.read_csv("data/ace_human_curated.csv")
print(f"{len(curated):,} molecules")
curated[["inchikey", "smiles", "pactivity", "activity", "evidence"]].head()

Not every molecule has a measured number. Some are known to be inactive only because a
paper said so, or because the experiment never reached 50% inhibition. Those are perfectly
good **labels**, but they are not usable **values**, so they can train the classifier and
not the regressor.

In [ ]:
curated["evidence"].value_counts().to_frame("molecules")

## 2. Look at the data

Before training anything, look at what the model will learn from. A model can only be as
good as its data, and a few plots often reveal problems that no metric will show later.

We start with the **class balance**: how many molecules inhibit ACE1 and how many do not.

In [ ]:
stylia.set_format("slide")
stylia.set_style("ersilia")
nc = stylia.NamedColors()

counts = curated["activity"].value_counts().rename({0: "inactive", 1: "active"})
counts = counts.reindex(["inactive", "active"])  # the order the colours below assume

fig, axs = stylia.create_figure(1, 1, width=0.3, height=0.3)
ax = axs.next()
ax.bar(counts.index, counts.values, color=[nc.blue, nc.yellow])
stylia.label(ax, xlabel="", ylabel="Molecules",
             title=f"{counts['active'] / counts.sum():.0%} of the molecules are active")

Two thirds of the molecules are active. That is **class imbalance**, and it matters in two
ways.

First, accuracy becomes misleading: a model that called every single molecule active would
already be right two thirds of the time while having learned nothing. We will use metrics
that do not fall for this.

Second, it is not an accident. Nobody publishes a paper about a molecule that does nothing,
so databases are full of compounds that worked. The model therefore sees a world with far
more inhibitors in it than the real world has.

> **Note:** The project plan lists "not enough data" as a risk for this project. With about
> a thousand molecules that risk is real, and it is the reason we lean on cross-validation
> in section 6 rather than trusting a single split.

Next, the **potency distribution**. The dashed line is the cutoff that separates the two
classes, at 1 uM.

In [ ]:
measured = curated[curated["evidence"] == "measured"]
fig, axs = stylia.create_figure(1, 1)
ax = axs.next()
ax.hist(measured["pactivity"], bins=40, range=(2.5, 11), color=nc.yellow)
ax.axvline(THRESHOLD, color=nc.pink, linestyle="--")
stylia.label(ax, xlabel="pActivity", ylabel="Molecules",
             title=f"Measured potency (median {measured['pactivity'].median():.2f})")

Size can give a model an easy shortcut. If the actives were simply bigger than the
inactives, the model could learn "big means active" and nothing about chemistry. We compute
the **molecular weight** of every molecule to check.

In [ ]:
from rdkit import Chem
from rdkit.Chem import Descriptors

curated["mw"] = [Descriptors.MolWt(Chem.MolFromSmiles(smi)) for smi in curated["smiles"]]
curated.groupby("activity")["mw"].describe().round(0)

The histogram shows the two classes side by side, with the same colours as the bar chart
above.

In [ ]:
fig, axs = stylia.create_figure(1, 1)
ax = axs.next()
for label, color, name in [(0, nc.blue, "inactive"), (1, nc.yellow, "active")]:
    weights = curated.loc[curated["activity"] == label, "mw"]
    ax.hist(weights, bins=40, range=(0, 1000), histtype="stepfilled", alpha=0.6,
            color=color, label=f"{name} (median {weights.median():.0f})")
ax.legend()
stylia.label(ax, xlabel="Molecular weight (g/mol)", ylabel="Molecules",
             title="Molecular weight of actives and inactives")

The two distributions sit almost on top of each other: the medians differ by about
ten g/mol, which is nothing. So size carries no information about whether a molecule inhibits
ACE1, and the model cannot take that shortcut. It has to use the actual chemistry.

Last, the **chemical series**. Medicinal chemists usually make many variations of one core
structure. That core is called the **scaffold** (or Bemis-Murcko scaffold): the rings of a
molecule and the chains connecting them, with every side group removed. We will need the
scaffolds in section 5, so we compute them now.

In [ ]:
curated["scaffold"] = modelling.murcko_scaffolds(curated["smiles"])
series = curated["scaffold"].value_counts()
print(f"{len(curated):,} molecules share {len(series):,} scaffolds")
print(f"{(series == 1).sum():,} scaffolds appear only once")
series.head(5).to_frame("molecules")

This dataset is unusually **diverse**: a quarter of the molecules are the only example of
their scaffold. Compare that with a dataset built by optimising one chemical series, where
a handful of scaffolds would cover almost everything.

Diversity is good news and bad news. Good, because the model sees many kinds of molecule.
Bad, because with about a thousand molecules spread over hundreds of series, it sees very
few examples of each.

> **Exercise:** Paste the most common scaffold into a structure viewer (for example
> https://molview.org). Many ACE1 inhibitors share a core because they were designed from
> the same starting point. Does the one you see look like the drugs the group found in the
> curation notebook?

## 3. Clean the molecules

The curation notebook already removed salts and duplicates, but a few entries are not really
small molecules: the heaviest here weighs over 2,000 g/mol, which is a peptide rather than
something anyone would swallow as a tablet.

The group is looking for **natural products taken by mouth**, which essentially never weigh
more than about 1,000 g/mol, so we drop everything above that line. It costs about 1% of this
dataset. The same filter is used by the purple group, so the two projects stay comparable.

In [ ]:
too_big = curated["mw"] > 1000
print(f"{too_big.sum()} molecules above 1,000 g/mol removed "
      f"({too_big.mean():.1%} of the data)")

curated = curated[~too_big].reset_index(drop=True)
print(f"{len(curated):,} molecules kept, {curated['activity'].mean():.1%} of them active")

> **Note:** With a dataset this small, every filter costs information. Nine molecules
> is a price worth paying to remove a part of chemical space the model would never be used
> on, but do not filter casually here: the group has about a thousand molecules in total.

## 4. Turn molecules into numbers

A model cannot read a SMILES string. It needs every molecule written as the same fixed list
of numbers, called **features**. Turning molecules into features is called **featurisation**.

We use a **Morgan fingerprint** (also called ECFP4). For every atom it looks at the small
fragment around it, up to two bonds away, and switches on one of 2,048 bits for that
fragment. Two molecules sharing many fragments share many bits, so similar molecules get
similar fingerprints.

This is one choice among many, and section 6 is about not having to make it by hand.

In [ ]:
X_class = modelling.morgan_fingerprints(curated["smiles"], radius=2, n_bits=2048)
y_class = curated["activity"].values
print(f"feature matrix: {X_class.shape[0]:,} molecules x {X_class.shape[1]:,} bits")
print(f"on average {X_class.sum(axis=1).mean():.0f} bits are switched on per molecule")

## 5. A random forest on fingerprints

### 5.1 A first model on a random split

To know whether a model works we must test it on molecules it has **never seen**. So we hold
back 20% of the molecules as a **test set** and train on the other 80%. The split is
stratified, meaning both parts keep the same share of actives.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_class, y_class, test_size=0.2, stratify=y_class, random_state=RANDOM_SEED)
print(f"training set: {len(y_train):,} molecules, {y_train.mean():.1%} active")
print(f"test set:     {len(y_test):,} molecules, {y_test.mean():.1%} active")

A **random forest** is a collection of decision trees. Each tree asks a series of yes-or-no
questions about the bits ("does the molecule contain this fragment?"), and the forest
averages the answers of all its trees.

`class_weight="balanced"` tells the model to care as much about the smaller class as the
larger one. With two thirds of the data active, that matters here.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

classifier = RandomForestClassifier(n_estimators=100, class_weight="balanced",
                                    n_jobs=-1, random_state=RANDOM_SEED)
classifier.fit(X_train, y_train)
proba = classifier.predict_proba(X_test)[:, 1]
print(f"predicted the probability of being active for {len(proba):,} test molecules")

The model gives each test molecule a **probability** of being active. We call it active when
that probability is 0.5 or more, and compare with the real labels. The metrics mean:

- **ROC-AUC**: how often an active molecule scores higher than an inactive one. 0.5 is a coin
  toss, 1 is perfect.
- **PR-AUC**: how precise the model stays as it finds more of the actives. Careful with this
  one: a model guessing at random scores the share of actives, which is about 0.66 here, not
  0.5. Only the amount above 0.66 is real skill.
- **Balanced accuracy**: the average of the share of actives and the share of inactives it
  gets right. Unlike plain accuracy, always calling "active" scores 0.5.
- **Precision**: of the molecules it calls active, how many really are.
- **Recall**: of the real actives, how many it finds.

In [ ]:
scores = modelling.classification_metrics(y_test, proba)
print(f"always guessing 'active' would give PR-AUC {y_test.mean():.3f}")
pd.Series(scores).round(3).to_frame("test set")

The **ROC curve** shows the trade-off behind ROC-AUC. Moving along the curve lowers the
probability needed to call a molecule active: the model finds more actives (up) but also
calls more inactives active by mistake (right). The diagonal is a coin toss.

Next to it, the same predictions split by the real class. Each dot is one test
molecule, placed at the probability the model gave it, and the box covers the middle half
of each class. A good model pushes the actives up and the inactives down; the dots that
cross the dashed line at 0.5 are its mistakes.


In [ ]:
from sklearn.metrics import roc_curve

fpr, tpr, _ = roc_curve(y_test, proba)
fig, axs = stylia.create_figure(1, 2, width=0.75, height=0.5, width_ratios=[2, 1])
ax = axs.next()
ax.plot(fpr, tpr, color=nc.yellow)
ax.plot([0, 1], [0, 1], color=nc.gray, linestyle=":")
stylia.label(ax, xlabel="False positive rate", ylabel="True positive rate",
             title="ROC curve (test set)")
ax = axs.next()
modelling.plot_proba_by_class(ax, y_test, proba, colors=[nc.blue, nc.yellow])
stylia.label(ax, xlabel="", ylabel="Predicted probability of being active",
             title="Scores by class")

The **confusion matrix** counts the four possible outcomes: actives called active, actives
missed, inactives called inactive, and inactives wrongly called active.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

fig, axs = stylia.create_figure(1, 1, width=0.5, height=0.5)
ax = axs.next()
ConfusionMatrixDisplay.from_predictions(
    y_test, (proba >= 0.5).astype(int), display_labels=["inactive", "active"],
    cmap="YlOrBr", colorbar=False, ax=ax)
ax.grid(False)
stylia.label(ax, xlabel="Predicted", ylabel="Real", title="Confusion matrix (test set)")

> **Note:** The test set holds only about 200 molecules, so each cell of this matrix
> rests on a small count and would move if we picked a different split. That is exactly why
> the next section repeats the whole thing five times.

### 5.2 Random or scaffold split: a fairer test

The split above was random. Because molecules come in series, a random split puts close
relatives of many test molecules into the training set. The model has then seen something
very similar before, and the test is easy.

That is not how this model will be used. The group wants to screen **indoles and xanthones**,
natural products that look nothing like the peptide-derived drugs that dominate ACE1 data. A
fairer test is a **scaffold split**: every molecule of a series goes to the same side, so the
test molecules have scaffolds the model has never seen.

We also switch to **5-fold cross-validation**: the data is cut into five parts and the model
is trained five times, each time tested on a different part. With a dataset this small that
matters, because a single split is easily a lucky one.

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold

random_folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
scaffold_folds = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
cv_class = pd.concat({
    "random": modelling.cross_validate(classifier, X_class, y_class, random_folds),
    "scaffold": modelling.cross_validate(classifier, X_class, y_class, scaffold_folds,
                                         groups=curated["scaffold"]),
}, names=["split"])
cv_class.groupby("split").agg(["mean", "std"]).round(3)

The scaffold split scores clearly lower, and its fold-to-fold spread (the `std`
column) is wide. Both facts come from the same place: a thousand molecules spread over
hundreds of series is not much to learn from. The scaffold number is the honest one.

## 6. LazyQSAR: let the computer choose

Everything above rests on one choice we made by hand in section 4: the Morgan fingerprint.
There are many other ways to describe a molecule, and there is no way to know in advance
which suits a given dataset.

**LazyQSAR** is an Ersilia tool that does that search for you. In `slow` mode it tries up to
five ways of describing a molecule, including some based on neural networks trained on
millions of molecules. It first discards any description that does not suit these molecules,
then keeps the ones that work best on the training data, fits a model to each and combines
them.

> **Note:** This is the slowest cell in the notebook. The first run also downloads about
> 900 MB of model weights and a reference library, so allow roughly ten to twenty minutes in
> Colab. Nothing is wrong if it sits quiet for a while.

To keep it honest we compare on the **scaffold split**, the harder one, and we give both
models exactly the same training and test molecules: the first fold of the same split used
above.

In [ ]:
train_rows, test_rows = next(
    scaffold_folds.split(X_class, y_class, groups=curated["scaffold"]))
smiles_train = curated["smiles"].iloc[train_rows].tolist()
smiles_test = curated["smiles"].iloc[test_rows].tolist()
y_fold_train, y_fold_test = y_class[train_rows], y_class[test_rows]
print(f"training on {len(train_rows):,} molecules, testing on {len(test_rows):,}")

LazyQSAR takes the molecules as SMILES rather than as a feature matrix, because
choosing the features is its job. `mode="slow"` is what tries all five descriptions;
`mode="fast"` would use fingerprints only and finish in seconds.

In [ ]:
from lazyqsar.qsar import LazyClassifierQSAR

lazy = LazyClassifierQSAR(mode="slow")
lazy.fit(smiles_train, y_fold_train)
print(f"descriptions kept: {lazy.descriptor_types}")

Watch the messages above: one of the five descriptions, `cddd`, is dropped before
any model is fitted, because too many of these molecules fall outside the chemical space it
was trained on. That check is worth knowing about. A description that has never seen
molecules like yours will produce numbers, and they will be meaningless.

Each kept description got its own model. Now we ask the combination for a probability on the
test molecules, and score the random forest on the very same molecules for comparison.

In [ ]:
lazy_proba = lazy.predict_proba(smiles_test)[:, 1]

forest = RandomForestClassifier(n_estimators=100, class_weight="balanced",
                                n_jobs=-1, random_state=RANDOM_SEED)
forest.fit(X_class[train_rows], y_fold_train)
forest_proba = forest.predict_proba(X_class[test_rows])[:, 1]

comparison = pd.DataFrame({
    "random forest": modelling.classification_metrics(y_fold_test, forest_proba),
    "LazyQSAR": modelling.classification_metrics(y_fold_test, lazy_proba),
})
comparison.round(3)

LazyQSAR also reports a **rank**: where a molecule falls among 50,000 reference
drug-like molecules, from 0 to 1. It orders molecules exactly as the probability does, but it
is easier to explain to a chemist: "this compound scores higher than 95% of drug-like
molecules" means more than "its probability is 0.8".

In [ ]:
ranks = lazy.predict_rank(smiles_test)[:, 1]
pd.DataFrame({"probability": lazy_proba[:5].round(3), "rank": ranks[:5].round(3),
              "really active": y_fold_test[:5]})

On this fold LazyQSAR comes out ahead of the fingerprint random forest, though by less
than the spread between folds in section 5.2, so treat it as a promising lead rather than a
settled result. Running the whole five-fold scaffold comparison for both models, which takes
about half an hour, gives LazyQSAR roughly 0.86 against the random forest's 0.83.

It costs something, too: minutes instead of seconds, and a large download. Whether that is
worth it depends on what the group does next with the model.

> **Exercise:** Rerun this section with `mode="fast"`. That uses fingerprints only, so it is
> the same information the random forest had. Does it still beat the random forest? If it
> does, the gain came from the modelling rather than from the molecular description.

## 7. Train the final model and save it

Everything so far was measurement: each model was trained on part of the data so that the
rest could test it. Now we train the final model on **all** the data, because more training
data makes a better model.

A final model has no test score of its own, since nothing is left to test it on. The numbers
that belong with it are the cross-validation scores from sections 5 and 6, so write those
down next to the file.

We keep LazyQSAR, the better of the two. This takes as long as the fit in section 6.

In [ ]:
final_model = LazyClassifierQSAR(mode="slow")
final_model.fit(curated["smiles"].tolist(), y_class)
print(f"trained on all {len(y_class):,} molecules")
print(f"descriptions kept: {final_model.descriptor_types}")

A LazyQSAR model is saved as a folder rather than a single file, so we zip it before
downloading. Do keep the download: a Colab runtime is deleted when it disconnects, and the
folder goes with it. Upload it to the group's Drive folder afterwards.

In [ ]:
import shutil

final_model.save("outputs/yellow_ace1_lazyqsar")
archive = shutil.make_archive("outputs/yellow_ace1_lazyqsar", "zip",
                              "outputs/yellow_ace1_lazyqsar")
modelling.download_file(archive)

To use the model later, unzip it, load it and hand it SMILES directly. LazyQSAR
remembers which molecular descriptions it chose, so unlike the random forest you do not have
to recreate the featurisation yourself.

In [ ]:
reloaded = LazyClassifierQSAR.load("outputs/yellow_ace1_lazyqsar")
examples = {"captopril (an ACE1 drug)": "CC(CS)C(=O)N1CCCC1C(=O)O",
            "lisinopril (an ACE1 drug)": "NCCCCC(NC(CCc1ccccc1)C(=O)O)C(=O)N1CCCC1C(=O)O",
            "ethanol": "CCO"}
pd.DataFrame({"molecule": list(examples),
              "probability of being active":
                  reloaded.predict_proba(list(examples.values()))[:, 1].round(3)})

Both drugs score higher than the solvent, which is the least we should ask. Take
little comfort from it, though: captopril and lisinopril are both in the training data, so
this shows how to call the model, not how well it works. The honest number is the
scaffold-split one.

> **Exercise:** Replace the examples with indoles and xanthones the group is
> interested in, and look at both the probability and the rank. Remember that a molecule
> unlike anything in the training set gets a prediction the model has no real basis for,
> however confident the number looks.

## Summary

- You looked at the data, then removed the few molecules above 1,000 g/mol. Two thirds of
  what remains is active, spread over hundreds of scaffolds with a quarter of them unique.
- A random forest on 2,048-bit Morgan fingerprints reaches a ROC-AUC of about 0.90 on a
  random split and about 0.83 on a scaffold split. The scaffold number is the fair one.
- LazyQSAR, which picks the molecular description for you, came out ahead on the same
  scaffold fold: better, but by less than the spread between folds, and much slower.
- You trained the final LazyQSAR model on all the data and downloaded it, with the
  cross-validation scores as the numbers to quote alongside.

**Next:** check whether indoles and xanthones fall inside the applicability domain of this
model before trusting any prediction made on them.